# Parent Document Retriever

**문서 검색과 문서 분할의 균형 잡기**

문서 검색 과정에서 문서를 적절한 크기의 조각(청크)으로 나누는 것은 다음의 **상충되는 두 가지 중요한 요소를 고려**해야 합니다.

1. 작은 문서를 원하는 경우: 이렇게 하면 문서의 임베딩이 그 의미를 가장 정확하게 반영할 수 있습니다. 문서가 너무 길면 임베딩이 의미를 잃어버릴 수 있습니다.
2. 각 청크의 맥락이 유지되도록 충분히 긴 문서를 원하는 경우입니다.

**`ParentDocumentRetriever`의 역할**

이 두 요구 사항 사이의 균형을 맞추기 위해 `ParentDocumentRetriever`라는 도구가 사용됩니다. 이 도구는 문서를 작은 조각으로 나누고, 이 조각들을 관리합니다. 검색을 진행할 때는, 먼저 이 작은 조각들을 찾아낸 다음, 이 조각들이 속한 원본 문서(또는 더 큰 조각)의 식별자(ID)를 통해 전체적인 맥락을 파악할 수 있습니다.

여기서 '부모 문서'란, 작은 조각이 나누어진 원본 문서를 말합니다. 이는 전체 문서일 수도 있고, 비교적 큰 다른 조각일 수도 있습니다. 이 방식을 통해 문서의 의미를 정확하게 파악하면서도, 전체적인 맥락을 유지할 수 있게 됩니다.

**정리**

- **문서 간의 계층 구조 활용**: `ParentDocumentRetriever`는 문서 검색의 효율성을 높이기 위해 문서 간의 계층 구조를 활용합니다.
- **검색 성능 향상**: 관련성 높은 문서를 빠르게 찾아내며, 주어진 질문에 대한 가장 적합한 답변을 제공하는 문서를 효과적으로 찾아낼 수 있습니다.
  문서를 검색할 때 자주 발생하는 두 가지 상충되는 요구 사항이 있습니다:


## ParentDocumentRetriever 
**LangChain에서 문서 분할과 검색을 결합하여 검색 품질을 개선하는 고급 검색 기법**입니다. 이 리트리버는 검색의 정확도와 정보의 완전성이라는 두 가지 목표를 동시에 달성하기 위해 '부모 문서(Parent Document)'와 '자식 청크(Child Chunks)'라는 개념을 사용합니다.

---

### 작동 원리

`ParentDocumentRetriever`는 크게 두 단계로 작동합니다.

1. **문서 분할 및 임베딩**:
    
    - **부모 문서**: 원본의 크고 완전한 문서들입니다. 이 문서들은 벡터스토어에 직접 저장되지 않고, 별도의 저장소(`docstore`)에 보관됩니다.
        
    - **자식 청크**: 부모 문서를 작은 조각(청크)들로 나눈 것입니다. 이 자식 청크들이 실제 **벡터스토어**에 임베딩되어 저장됩니다.
        
2. **검색 및 재구성**:
    
    - 사용자가 질의를 입력하면, `ParentDocumentRetriever`는 먼저 **자식 청크**에 대해 벡터 유사도 검색을 수행합니다.
        
    - 검색 결과로 가장 관련성이 높은 몇 개의 자식 청크들이 반환됩니다.
        
    - 이 자식 청크들은 자신들의 **부모 문서 ID**를 가지고 있습니다. 리트리버는 이 ID를 사용하여 `docstore`에 저장된 **원본 부모 문서**를 불러옵니다.
        
    - 최종적으로, 검색된 **자식 청크를 포함하고 있는 전체 부모 문서**를 언어 모델(LLM)에게 전달합니다.
        

### 왜 ParentDocumentRetriever를 사용하는가?

이 방식은 기존의 검색 기법이 가진 문제점을 해결합니다.

- **정확도(Accuracy)**: 기존 검색기는 문서를 너무 크게 나누면 검색 정확도가 떨어지고, 너무 작게 나누면 문맥(context)이 손실되는 문제가 있었습니다. `ParentDocumentRetriever`는 **작은 청크를 사용하여 정밀한 검색**을 수행하고, 동시에 **전체 부모 문서를 제공하여 풍부한 문맥**을 보장합니다.
    
- **완전성(Completeness)**: LLM에 전체 부모 문서를 제공함으로써, 답변에 필요한 모든 정보가 포함될 가능성을 높입니다. 작은 청크만으로는 중요한 정보가 누락될 수 있는 위험을 줄입니다.
    

**요약하자면**, `ParentDocumentRetriever`는 **'정확한 탐색(작은 청크)'**과 **'풍부한 정보 제공(전체 문서)'**을 결합하여, 검색-증강 생성(RAG) 시스템의 성능을 향상시키는 강력한 도구입니다.

노래의 한 소절을 들려주고 노래를 맞추는 방식은 ParentDocumentRetriever의 작동 방식과 정확하게 일치하는 좋은 비유가 될 수 있습니다.

비유를 통한 설명
한 소절 (자식 청크): 노래의 한 소절은 노래 전체에서 가장 특징적이고 기억하기 쉬운 작은 부분입니다. 이 소절만 듣고도 어떤 노래인지 알 수 있죠. 이는 ParentDocumentRetriever에서 **정확한 검색을 위해 사용되는 작은 '자식 청크'**와 같습니다.

노래 전체 (부모 문서): 노래 전체는 한 소절이 속한 원본이자 완전한 형태입니다. 한 소절을 듣고 노래 제목을 맞춘 뒤, "아, 이 노래는 이 가수가 부른 2집 앨범의 3번 트랙이구나"와 같이 노래 전체와 관련된 풍부한 정보를 얻게 됩니다. 이는 **자식 청크를 통해 검색된 '부모 문서'**에 해당합니다.

ParentDocumentRetriever와의 연결점
정확한 검색: 한 소절(자식 청크)만으로 노래를 정확하게 찾을 수 있듯이, ParentDocumentRetriever는 작은 청크를 사용하여 유사도 검색의 정확성을 높입니다.

풍부한 문맥 제공: 노래 제목을 맞춘 후, 우리는 노래 전체(부모 문서)를 듣고 더 넓은 문맥과 풍부한 정보를 얻게 됩니다. ParentDocumentRetriever도 검색된 청크의 부모 문서를 LLM에 제공함으로써, 답변에 필요한 충분한 문맥을 보장합니다.

따라서, 노래의 한 소절을 듣고 노래를 맞추는 것은 ParentDocumentRetriever가 작은 단위로 정확하게 검색한 후, 그 작은 단위가 속한 전체 문맥을 가져오는 과정을 매우 직관적으로 설명해주는 비유입니다.

여러 개의 텍스트 파일을 로드하기 위해 `TextLoader` 객체를 생성하고 데이터를 로드합니다.


In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH10-Retriever")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


In [3]:
from langchain.storage import InMemoryStore
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ParentDocumentRetriever

In [5]:
loaders = [
    # 파일을 로드합니다.
    TextLoader("./data/appendix-keywords.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    # 로더를 사용하여 문서를 로드하고 docs 리스트에 추가합니다.
    docs.extend(loader.load())

## 전체 문서 검색

이 모드에서는 전체 문서를 검색하고자 합니다. 따라서 `child_splitter` 만 지정하도록 하겠습니다.

- 나중에는 `parent_splitter` 도 지정하여 결과를 비교해 보겠습니다.


In [7]:
# 자식 분할기를 생성합니다.
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

# DB를 생성합니다.
vectorstore = Chroma(
    collection_name="full_documents", embedding_function=OpenAIEmbeddings()
)

store = InMemoryStore()

# Retriever 를 생성합니다.
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


`retriever.add_documents(docs, ids=None)` 함수로 문서목록을 추가합니다.

- `ids` 가 `None` 이면 자동으로 생성됩니다.
- `add_to_docstore=False` 로 설정시 document 를 중복으로 추가하지 않습니다. 단, 중복을 체크하기 위한 `ids` 값이 필수 값으로 요구됩니다.


In [8]:
# 문서를 검색기에 추가합니다. docs는 문서 목록이고, ids는 문서의 고유 식별자 목록입니다.
retriever.add_documents(docs, ids=None, add_to_docstore=True)

이 코드는 두 개의 키를 반환해야 합니다. 그 이유는 우리가 두 개의 문서를 추가했기 때문입니다.


- `store` 객체의 `yield_keys()` 메서드를 호출하여 반환된 키(key) 값들을 리스트로 변환합니다.


In [9]:
# 저장소의 모든 키를 리스트로 반환합니다.
list(store.yield_keys())

['12bdcf0a-5a2c-4064-9022-2623122b1793']

이제 벡터 스토어 검색 기능을 호출해 보겠습니다.

우리가 작은 청크(chunk)들을 저장하고 있기 때문에, 검색 결과로 작은 청크들이 반환되는 것을 확인할 수 있을 것입니다.


`vectorstore` 객체의 `similarity_search` 메서드를 사용하여 유사도 검색을 수행합니다.


In [10]:
# 유사도 검색을 수행합니다.
sub_docs = vectorstore.similarity_search("Word2Vec")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


`sub_docs[0].page_content`를 출력합니다.


In [11]:
# sub_docs 리스트의 첫 번째 요소의 page_content 속성을 출력합니다.
print(sub_docs[0].page_content)

Word2Vec
정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성


이제 전체 retriever에서 검색해 보겠습니다. 이 과정에서는 작은 청크(chunk)들이 위치한 **문서를 반환** 하기 때문에 상대적으로 큰 문서들이 반환될 것입니다.


`retriever` 객체의 `invoke()` 메서드를 사용하여 쿼리와 관련된 문서를 검색합니다.


In [12]:
# 문서를 검색하여 가져옵니다.
retrieved_docs = retriever.invoke("Word2Vec")

검색된 문서(`retrieved_docs[0]`)의 일부 내용을 출력합니다.


In [13]:
# 검색된 문서의 문서의 페이지 내용의 길이를 출력합니다.
print(
    f"문서의 길이: {len(retrieved_docs[0].page_content)}",
    end="\n\n=====================\n\n",
)

# 문서의 일부를 출력합니다.
print(retrieved_docs[0].page_content[2000:2500])

문서의 길이: 5704


데이터 저장과 처리를 혁신하는 것은 디지털 변환의 예입니다.
연관키워드: 혁신, 기술, 비즈니스 모델

Crawling
정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec
정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성

LLM (Large Language Model)
정의: LLM은 대규모의 텍스트 데이터로 훈련된 큰 규모의 언어 모델을 의미합니다. 이러한 모


## 더 큰 Chunk 의 크기를 조절

<u>원본 문서가 너무 길어 통째로 부모 문서로 사용하기 부담스러울 때, parent_splitter를 사용하여 원본 문서를 관리 가능한 크기의 부모 문서들로 먼저 나눕니다.</u>

이전의 결과처럼 **전체 문서가 너무 커서 있는 그대로 검색하기에는 부적합** 할 수 있습니다.

이런 경우, 실제로 우리가 하고 싶은 것은 먼저 원시 문서를 더 큰 청크로 분할한 다음, 더 작은 청크로 분할하는 것입니다.

그런 다음 작은 청크들을 인덱싱하지만, 검색 시에는 더 큰 청크를 검색합니다 (그러나 여전히 전체 문서는 아닙니다).


- `RecursiveCharacterTextSplitter`를 사용하여 부모 문서와 자식 문서를 생성합니다.
  - 부모 문서는 `chunk_size`가 1000으로 설정되어 있습니다.
  - 자식 문서는 `chunk_size`가 200으로 설정되어 있으며, 부모 문서보다 작은 크기로 생성됩니다.


In [15]:
# 부모 문서를 생성하는 데 사용되는 텍스트 분할기입니다.
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
# 자식 문서를 생성하는 데 사용되는 텍스트 분할기입니다.
# 부모보다 작은 문서를 생성해야 합니다.
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
# 자식 청크를 인덱싱하는 데 사용할 벡터 저장소입니다.
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# 부모 문서의 저장 계층입니다.
store = InMemoryStore()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


`ParentDocumentRetriever`를 초기화하는 코드입니다.

- `vectorstore` 매개변수는 문서 벡터를 저장하는 벡터 저장소를 지정합니다.
- `docstore` 매개변수는 문서 데이터를 저장하는 문서 저장소를 지정합니다.
- `child_splitter` 매개변수는 하위 문서를 분할하는 데 사용되는 문서 분할기를 지정합니다.
- `parent_splitter` 매개변수는 상위 문서를 분할하는 데 사용되는 문서 분할기를 지정합니다.

`ParentDocumentRetriever`는 계층적 문서 구조를 처리하며, 상위 문서와 하위 문서를 별도로 분할하고 저장합니다. 이를 통해 검색 시 상위 문서와 하위 문서를 효과적으로 활용할 수 있습니다.


In [16]:
retriever = ParentDocumentRetriever(
    # 벡터 저장소를 지정합니다.
    vectorstore=vectorstore,
    # 문서 저장소를 지정합니다.
    docstore=store,
    # 하위 문서 분할기를 지정합니다.
    child_splitter=child_splitter,
    # 상위 문서 분할기를 지정합니다.
    parent_splitter=parent_splitter,
)

`retriever` 객체에 `docs`를 추가합니다. `retriever`가 검색할 수 있는 문서 집합에 새로운 문서들을 추가하는 역할을 합니다.


In [17]:
retriever.add_documents(docs)  # 문서를 retriever에 추가합니다.

이제 문서의 수가 훨씬 더 많아진 것을 볼 수 있습니다. 이는 더 큰 청크(chunk)들입니다.


In [19]:
# 저장소에서 키를 생성하고 리스트로 변환한 후 길이를 반환합니다.
len(list(store.yield_keys()))

7

기본 벡터 저장소가 여전히 작은 청크를 검색하는지 확인해 보겠습니다.


`vectorstore` 객체의 `similarity_search` 메서드를 사용하여 유사도 검색을 수행합니다.


In [20]:
# 유사도 검색을 수행합니다.
sub_docs = vectorstore.similarity_search("Word2Vec")
# sub_docs 리스트의 첫 번째 요소의 page_content 속성을 출력합니다.
print(sub_docs[0].page_content)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Word2Vec
정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성


이번에는 `retriever` 객체의 `invoke()` 메서드를 사용하여 문서를 검색합니다.


In [21]:
# 문서를 검색하여 가져옵니다.
retrieved_docs = retriever.invoke("Word2Vec")

# 검색된 문서의 첫 번째 문서의 페이지 내용의 길이를 반환합니다.
print(retrieved_docs[0].page_content)

Transformer
정의: 트랜스포머는 자연어 처리에서 사용되는 딥러닝 모델의 한 유형으로, 주로 번역, 요약, 텍스트 생성 등에 사용됩니다. 이는 Attention 메커니즘을 기반으로 합니다.
예시: 구글 번역기는 트랜스포머 모델을 사용하여 다양한 언어 간의 번역을 수행합니다.
연관키워드: 딥러닝, 자연어 처리, Attention

HuggingFace
정의: HuggingFace는 자연어 처리를 위한 다양한 사전 훈련된 모델과 도구를 제공하는 라이브러리입니다. 이는 연구자와 개발자들이 쉽게 NLP 작업을 수행할 수 있도록 돕습니다.
예시: HuggingFace의 Transformers 라이브러리를 사용하여 감정 분석, 텍스트 생성 등의 작업을 수행할 수 있습니다.
연관키워드: 자연어 처리, 딥러닝, 라이브러리

Digital Transformation
정의: 디지털 변환은 기술을 활용하여 기업의 서비스, 문화, 운영을 혁신하는 과정입니다. 이는 비즈니스 모델을 개선하고 디지털 기술을 통해 경쟁력을 높이는 데 중점을 둡니다.
예시: 기업이 클라우드 컴퓨팅을 도입하여 데이터 저장과 처리를 혁신하는 것은 디지털 변환의 예입니다.
연관키워드: 혁신, 기술, 비즈니스 모델

Crawling
정의: 크롤링은 자동화된 방식으로 웹 페이지를 방문하여 데이터를 수집하는 과정입니다. 이는 검색 엔진 최적화나 데이터 분석에 자주 사용됩니다.
예시: 구글 검색 엔진이 인터넷 상의 웹사이트를 방문하여 콘텐츠를 수집하고 인덱싱하는 것이 크롤링입니다.
연관키워드: 데이터 수집, 웹 스크래핑, 검색 엔진

Word2Vec
정의: Word2Vec은 단어를 벡터 공간에 매핑하여 단어 간의 의미적 관계를 나타내는 자연어 처리 기술입니다. 이는 단어의 문맥적 유사성을 기반으로 벡터를 생성합니다.
예시: Word2Vec 모델에서 "왕"과 "여왕"은 서로 가까운 위치에 벡터로 표현됩니다.
연관키워드: 자연어 처리, 임베딩, 의미론적 유사성
